In [ ]:
# Import smoke test — multipart >= 1.0 is a single-file module (multipart.py,
# no __path__), so this cell checks the top-level import and version.
import multipart

print(f"multipart.__version__ = {multipart.__version__}")
assert not hasattr(multipart, '__path__'), (
    'multipart is now a package; update this check to cover its submodules'
)
print(f"OK — {multipart.__name__} imported (flat module, no submodules)")


In [ ]:
# Example 1: parse_options_header — split a Content-Type header into value + options.
from multipart import parse_options_header

header = 'multipart/form-data; boundary=----WebKitFormBoundary123; charset=utf-8'
value, options = parse_options_header(header)

print(f"value   = {value!r}")
print(f"options = {options!r}")

assert value == 'multipart/form-data'
assert options.get('boundary') == '----WebKitFormBoundary123'
assert options.get('charset') == 'utf-8'


In [ ]:
# Example 2: PushMultipartParser — push-based streaming parser (v1.x API).
from multipart import PushMultipartParser, MultipartSegment

boundary = 'BOUNDARY'
body = (
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="field1"\r\n'
    '\r\n'
    'value1\r\n'
    f'--{boundary}--\r\n'
).encode('utf-8')

events = []
with PushMultipartParser(boundary) as parser:
    for event in parser.parse(body):
        events.append(event)

print(f"events: {[type(e).__name__ for e in events]}")
segments = [e for e in events if isinstance(e, MultipartSegment)]
assert len(segments) == 1
assert segments[0].name == 'field1'


In [ ]:
# Example 3: MultipartParser — iterate parts from a file-like stream.
import io
from multipart import MultipartParser

boundary = 'XBOUND'
body = (
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="alpha"\r\n'
    '\r\n'
    'hello\r\n'
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="beta"; filename="b.txt"\r\n'
    'Content-Type: text/plain\r\n'
    '\r\n'
    'file-contents\r\n'
    f'--{boundary}--\r\n'
).encode('utf-8')

parser = MultipartParser(io.BytesIO(body), boundary)
parts = list(parser)
print(f"parts: {[(p.name, p.filename, p.value if not p.filename else '<file>') for p in parts]}")

assert len(parts) == 2
assert parts[0].name == 'alpha' and parts[0].value == 'hello'
assert parts[1].name == 'beta' and parts[1].filename == 'b.txt'


In [ ]:
# Example 4: parse_form_data — high-level WSGI-style form parsing.
import io
from multipart import parse_form_data

boundary = 'FORMBOUND'
body = (
    f'--{boundary}\r\n'
    'Content-Disposition: form-data; name="username"\r\n'
    '\r\n'
    'alice\r\n'
    f'--{boundary}--\r\n'
).encode('utf-8')

environ = {
    'REQUEST_METHOD': 'POST',
    'CONTENT_TYPE': f'multipart/form-data; boundary={boundary}',
    'CONTENT_LENGTH': str(len(body)),
    'wsgi.input': io.BytesIO(body),
}

forms, files = parse_form_data(environ)
print(f"forms = {dict(forms)}")
print(f"files = {dict(files)}")
assert forms['username'] == 'alice'


In [ ]:
# Example 5: PushMultipartParser handles malformed input by raising MultipartError.
from multipart import PushMultipartParser, MultipartError

boundary = 'BOUND'
# Body has wrong boundary marker — should error out.
bad_body = b'--WRONGBOUNDARY\r\nContent-Disposition: form-data; name="x"\r\n\r\nv\r\n--WRONGBOUNDARY--\r\n'

raised = False
try:
    with PushMultipartParser(boundary) as parser:
        for _ in parser.parse(bad_body):
            pass
        # Force end-of-stream signal
        for _ in parser.parse(b''):
            pass
except MultipartError as e:
    raised = True
    print(f"got expected MultipartError: {e}")

assert raised, 'expected MultipartError for mismatched boundary'
